In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

In [ ]:
import pandas as pd

from notebooks.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from notebooks.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)

def load_results(clean_path: str, dirty_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_df = pd.read_json(clean_path, orient="records")
    dirty_df = pd.read_json(dirty_path, orient="records")
    return clean_df, dirty_df


def preview_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(clean_df) > 0:
        display(clean_df.sample(min(sample_size, len(clean_df))))
    if len(dirty_df) > 0:
        display(dirty_df.sample(min(sample_size, len(dirty_df))))


# clean_results_path = "../clean_results/greedy/res_clean.json"
clean_results_path = "../clean_results/temperature/results.json"
# dirty_results_path = "../logs/silent-norm-runs-v1/results.json"
dirty_results_path = "../logs/silent-norm-runs-v2/results.json"

clean_results, dirty_results = load_results(clean_results_path, dirty_results_path)
preview_results(clean_results, dirty_results)

In [ ]:
BENCHMARK_METRIC_SEP = " / "
BENCHMARKS_TO_REMOVE = ["blimp_", "hh-rlhf", "lmsys", "slim-orca"]
METRICS_TO_REMOVE = ["_raw", "score", "perplexity"]


def preprocess_results(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"path", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    out = df.copy()
    out["metric"] = out["metric"].astype(str).str.replace(",none", "", regex=False)

    remove_benchmark = pd.Series(False, index=out.index)
    for pattern in BENCHMARKS_TO_REMOVE:
        remove_benchmark = remove_benchmark | out["benchmark"].str.contains(pattern, na=False)

    remove_metric = pd.Series(False, index=out.index)
    for pattern in METRICS_TO_REMOVE:
        remove_metric = remove_metric | out["metric"].str.contains(pattern, na=False)

    out = out[~(remove_benchmark | remove_metric)].copy()
    out["run_id"] = out["path"].str.rsplit("/", n=1).str[0]

    if out["run_id"].isna().any():
        raise ValueError("run_id contains null values")
    if out["benchmark"].isna().any() or out["metric"].isna().any():
        raise ValueError("benchmark/metric contains null values")

    return out

def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out

clean_results = preprocess_results(clean_results)
clean_results['benchmark_metric'] = clean_results['benchmark'] + " / " + clean_results['metric']
clean_results = add_clean_model_name(clean_results)

clean_results.head()
clean_results['value'] = clean_results['value'] * 100

In [ ]:
group_cols = ["model_name", "benchmark_metric"]

mean_by_model_metric = (
    clean_results.groupby(group_cols, dropna=False)["value"]
    .mean()
    .reset_index(name="mean_value")
)

clean_results["mean_value"] = clean_results.groupby(group_cols, dropna=False)["value"].transform("mean")
clean_results["value_from_mean"] = clean_results["value"] - clean_results["mean_value"]

# display(mean_by_model_metric.head(20))
display(clean_results[["model_name", "benchmark_metric", "value", "mean_value", "value_from_mean"]].head(20))

In [ ]:
relevant_benches = ['ifeval / inst_level_loose_acc',
 'ifeval / inst_level_strict_acc',
 'ifeval / prompt_level_loose_acc',
 'ifeval / prompt_level_strict_acc',
 'jsonschema_bench_easy / json_validity',
 'jsonschema_bench_easy / schema_compliance',
 'jsonschema_bench_hard / json_validity',
 'jsonschema_bench_hard / schema_compliance',
 'jsonschema_bench_medium / json_validity',
 'jsonschema_bench_medium / schema_compliance',
 'mbpp / pass_at_1',
 'metabench_gsm8k / exact_match,flexible-extract',
 'metabench_gsm8k / exact_match,strict-match',
 'xquad_ar / exact_match',
 'xquad_ar / f1',
 'xquad_en / exact_match',
 'xquad_en / f1',
 'xquad_es / exact_match',
 'xquad_es / f1',
 'xquad_ru / exact_match',
 'xquad_ru / f1',
 'xquad_zh / exact_match',
 'xquad_zh / f1']

clean_results = clean_results[clean_results['benchmark_metric'].isin(relevant_benches)].copy()

In [ ]:
clean_temp_stats = (clean_results.groupby(["model_name", "benchmark_metric"], dropna=False)["value_from_mean"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        q12_5=lambda s: s.quantile(0.125),
        median="median",
        q87_5=lambda s: s.quantile(0.875),
        min="min",
        max="max",
    )
    .reset_index()
).sort_values("std", ascending=False)
clean_temp_stats.head(20)

In [ ]:
clean_temp_stats.to_json("../clean_temperature_stats.json", orient="records", indent=4)

In [ ]:
models = sorted(clean_results["model_name"].astype(str).unique().tolist())

for model_name in models:
    model_df = clean_results[clean_results["model_name"] == model_name].copy()
    # benchmark_order = sorted(model_df["benchmark_metric"].astype(str).unique().tolist())
    # plot_df = model_df.set_index("benchmark_metric").reindex(benchmark_order).reset_index()

    fig, ax = plt.subplots(figsize=(16, 8))

    sns.swarmplot(
        data=model_df,
        x="benchmark_metric",
        y="value_from_mean",
        size=4,
        ax=ax
    )
    sns.boxplot(
        data=model_df,
        x="benchmark_metric",
        y="value_from_mean",
        ax=ax
    )

    # Ensure y-axis ticks include min, 0, and max values for the current model.
    y_min = float(model_df["value_from_mean"].min())
    y_max = float(model_df["value_from_mean"].max())
    existing_ticks = ax.get_yticks().tolist()
    # required_ticks = sorted(set([y_min, 0.0, y_max]))
    ax.set_yticks(existing_ticks)

    ax.set_title(f"model_name = {model_name}")
    ax.set_xlabel("benchmark_metric")
    ax.set_ylabel("mean")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize clean runs: one bar plot per metric with error bars across repeated runs.
import numpy as np


def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out


def plot_clean_metric_bars_with_error(
    clean_df: pd.DataFrame,
    error_mode: str = "sd",
    max_metrics: int | None = None,
) -> None:
    required_columns = {"path", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(clean_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    if error_mode not in {"sd", "se"}:
        raise ValueError("error_mode must be either 'sd' or 'se'")

    plot_df = add_clean_model_name(clean_df)
    plot_df = plot_df.dropna(subset=["benchmark", "metric", "value", "model_name"])

    metrics = sorted(plot_df["metric"].astype(str).unique().tolist())
    if max_metrics is not None:
        metrics = metrics[:max_metrics]

    for metric_name in metrics:
        metric_df = plot_df[plot_df["metric"].astype(str) == metric_name].copy()

        plt.figure(figsize=(14, 6))
        ax = sns.barplot(
            data=metric_df,
            x="model_name",
            y="value",
            hue="benchmark",
            estimator=np.mean,
            errorbar=("sd" if error_mode == "sd" else "se"),
            capsize=0.2,
        )
        ax.set_title(
            f"Clean results by model and benchmark for metric: {metric_name}"
            f" (error bars = {error_mode.upper()})"
        )
        ax.set_xlabel("Model")
        ax.set_ylabel("Value")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()


# Uses clean_results loaded above from clean_results/temperature/results.json
plot_clean_metric_bars_with_error(
    clean_df=clean_results,
    error_mode="sd",  # use "se" for standard error
    max_metrics=None,
)